# News Source Reliability Pipeline

End-to-end NLP pipeline over all 16 news outlets (`SOURCE_SCOPE = "all"` — the primary reliability ranking; switch to `"wires"` for the validated four-wire deep-dive): scores the sentiment of every article, fact-checks macro claims against `macro.csv` ground truth, derives structural reliability signals (corroboration, coverage volume, rare-event coverage) and sentiment-vs-macro divergence, blends them into a per-source composite reliability score using fixed anchors rather than min-max (so scores don't depend on the other sources and small gaps stay small), and trains an MLflow-tracked logistic regression with TF-IDF headline features to predict which claims match the macro data.

In [0]:
import pandas as pd
import numpy as np

DATA_DIR = "/Workspace/Repos/DSDG Hackathon/DSDG Hackathon/Data"

news = pd.read_csv(f"{DATA_DIR}/news.csv")
macro = pd.read_csv(f"{DATA_DIR}/macro.csv")

# Source scope: "all" (all 16 outlets — the primary reliability ranking) or "wires"
# (the four wire services only — the audited deep-dive subset; see cell 8)
SOURCE_SCOPE = "all"
WIRE_SERVICES = {"Freight Signal", "World Trade Monitor", "Global Commodities Brief", "Merrow Maritime Daily"}
if SOURCE_SCOPE == "wires":
    news = news[news["source"].isin(WIRE_SERVICES)].reset_index(drop=True)
print(f"scope: {SOURCE_SCOPE} | articles: {len(news):,} across {news['source'].nunique()} sources")

# Map country names in headlines to macro.csv country codes
COUNTRY_MAP = {
    "Halvard": "HAL", "Kestria": "KES", "Valdmar": "VAL",
    "Orinoque": "ORI", "Aurelia": "AUR", "Sunda Coast": "SUN",
}

news["country_code"] = news["headline"].map(
    lambda h: next((code for name, code in COUNTRY_MAP.items() if name in h), None)
)

# Previous-week macro values per country (ground truth for week-over-week changes)
macro = macro.sort_values(["country", "week"]).reset_index(drop=True)
macro["pmi_prev"] = macro.groupby("country")["pmi"].shift(1)
macro["policy_rate_prev"] = macro.groupby("country")["policy_rate"].shift(1)
macro["tension_index_prev"] = macro.groupby("country")["tension_index"].shift(1)

# Join news to macro on week + country
news = news.merge(
    macro.drop(columns=["date"]),
    left_on=["week", "country_code"],
    right_on=["week", "country"],
    how="left",
).drop(columns=["country"])

print(f"news rows: {len(news):,} | macro rows: {len(macro):,} | sources: {news['source'].nunique()} | weeks: {news['week'].min()}-{news['week'].max()}")
print(f"articles with country identified from headline: {news['country_code'].notna().sum():,} ({news['country_code'].notna().mean():.1%})")
print(f"articles joined to macro ground truth: {news['pmi'].notna().sum():,}")

In [0]:
# Sentiment scoring with a finance/news lexicon (negative-to-positive score in [-1, 1]).
# Raw headline text is kept untransformed here: the ML stage consumes it via TF-IDF.
import re

SENTIMENT_METHOD = "custom finance/news lexicon"
LEXICON = {
        # positive
        "grows": 1.0, "up": 0.8, "rises": 1.0, "widens": 0.8, "expands": 1.0,
        "gains": 1.0, "steady": 0.4, "stable": 0.4, "strong": 1.0, "boost": 1.0,
        "invest": 0.8, "investment": 0.8, "infrastructure": 0.6, "opens": 0.8,
        "surplus": 1.0, "optimism": 1.0, "rebounds": 1.0, "recovery": 1.0,
        "record": 0.8, "approves": 0.8, "upgrade": 1.0, "improves": 1.0,
        "eases": 0.5, "calm": 0.8, "spending": 0.6,
        # negative
        "falls": -1.0, "down": -0.8, "drops": -1.0, "declines": -1.0, "slump": -1.2,
        "weak": -1.0, "weakens": -1.0, "delays": -0.8, "congestion": -1.0,
        "tensions": -1.2, "warning": -1.0, "shots": -1.0, "fired": -1.0,
        "dispute": -1.0, "unrest": -1.2, "shortfall": -1.0, "cuts": -0.8,
        "layoffs": -1.0, "protests": -1.0, "slide": -1.0, "plunge": -1.2,
        "losses": -1.0, "uncertainty": -0.8, "mixed": -0.3, "cautious": -0.5,
        "slowdown": -1.0, "shrinks": -0.8, "halts": -0.8, "suspends": -0.8,
    }

def score_text(text: str) -> float:
    tokens = re.findall(r"[a-z]+", text.lower())
    hits = [LEXICON[t] for t in tokens if t in LEXICON]
    if not hits:
        return 0.0
    raw = sum(hits) / (len(tokens) ** 0.5)
    return float(np.clip(raw, -1.0, 1.0))

news["headline_sentiment"] = news["headline"].map(score_text)
news["body_sentiment"] = news["body"].map(score_text)
# Headlines carry the claim; weight them above bodies
news["sentiment"] = 0.6 * news["headline_sentiment"] + 0.4 * news["body_sentiment"]

print(f"Sentiment method: {SENTIMENT_METHOD}")
print("\nMean article sentiment by source:")
print(news.groupby("source")["sentiment"].mean().sort_values().round(3).to_string())

In [0]:
# Classify each headline into a claim type and verify it against macro.csv ground truth

def classify_claim(headline: str):
    if "factory activity edges up" in headline:
        return "pmi_up"
    if "factory activity edges down" in headline:
        return "pmi_down"
    if "holds rates steady" in headline:
        return "rate_steady"
    if "Tensions rise" in headline:
        return "tension_up"
    return None

news["claim"] = news["headline"].map(classify_claim)


def claim_is_checkable(row) -> bool:
    c = row["claim"]
    if c in ("pmi_up", "pmi_down"):
        return pd.notna(row["pmi"]) and pd.notna(row["pmi_prev"])
    if c == "rate_steady":
        return pd.notna(row["policy_rate"]) and pd.notna(row["policy_rate_prev"])
    if c == "tension_up":
        return pd.notna(row["tension_index"]) and pd.notna(row["tension_index_prev"])
    return False


def claim_is_accurate(row) -> bool:
    c = row["claim"]
    if c == "pmi_up":
        return bool(row["pmi"] > row["pmi_prev"])
    if c == "pmi_down":
        return bool(row["pmi"] < row["pmi_prev"])
    if c == "rate_steady":
        return bool(row["policy_rate"] == row["policy_rate_prev"])
    if c == "tension_up":
        return bool(row["tension_index"] > row["tension_index_prev"])
    return None


news["checkable"] = news.apply(claim_is_checkable, axis=1)
news.loc[news["checkable"], "claim_accurate"] = news[news["checkable"]].apply(claim_is_accurate, axis=1)

chk = news[news["checkable"]]
print(f"Checkable claims: {len(chk):,} of {len(news):,} articles | overall accuracy: {chk['claim_accurate'].mean():.1%}")
print("\nAccuracy by claim type:")
print(chk.groupby("claim")["claim_accurate"].agg(["count", "mean"]).round(3).to_string())

# Per-source accuracy with shrinkage smoothing (pulls small samples toward the global rate,
# so regional papers with ~15 checkable stories aren't over-rewarded or over-punished)
acc = chk.groupby("source")["claim_accurate"].agg(["count", "mean"])
acc.columns = ["n_checkable", "raw_accuracy"]
prior = chk["claim_accurate"].mean()
acc["smoothed_accuracy"] = (acc["n_checkable"] * acc["raw_accuracy"] + 10 * prior) / (acc["n_checkable"] + 10)
display(acc.sort_values("smoothed_accuracy").round(3))

In [0]:
# Structural reliability signals per source
news["is_wire"] = news["source"].isin(WIRE_SERVICES)

# Corroboration: identical story (same week + headline + body) carried by more than one source
story_carriers = news.groupby(["week", "headline", "body"])["source"].nunique().rename("n_carriers").reset_index()
news = news.merge(story_carriers, on=["week", "headline", "body"], how="left")
news["corroborated"] = news["n_carriers"] > 1

# Rare/disruptive events (wire services carried ~70% of these in prior analysis)
RARE_KEYWORDS = ["Tensions", "Warning shots", "Congestion", "labor talks", "maintenance delays", "Calm returns"]
news["is_rare_event"] = news["headline"].str.contains("|".join(RARE_KEYWORDS), regex=True)

structural = news.groupby("source").agg(
    articles=("article_id", "count"),
    corroboration_rate=("corroborated", "mean"),
    rare_event_articles=("is_rare_event", "sum"),
).reset_index()
structural["volume_share"] = structural["articles"] / structural["articles"].sum()
structural["rare_event_share"] = structural["rare_event_articles"] / news["is_rare_event"].sum()

print(f"Total rare-event articles: {news['is_rare_event'].sum()}")
print(f"Wire services' share of rare-event coverage: {news.loc[news['is_rare_event'], 'is_wire'].mean():.0%}")
display(structural.sort_values("rare_event_share", ascending=False).round(3))

In [0]:
# Sentiment-vs-macro divergence: does a source's tone track what the macro data justifies?
# macro_context = PMI week-over-week change minus a scaled tension increase (higher = genuinely better week)
j = news[news["country_code"].notna() & news["pmi_prev"].notna()].copy()
j["macro_context"] = (j["pmi"] - j["pmi_prev"]) - 0.05 * (j["tension_index"] - j["tension_index_prev"])

rows = []
for src, g in j.groupby("source"):
    if len(g) < 20:
        continue  # too few country-tagged articles for a stable signal
    corr = g["sentiment"].corr(g["macro_context"])  # sentiment-macro correlation
    # macro sensitivity: tone in genuinely good weeks minus tone in genuinely bad weeks
    good = g.loc[g["macro_context"] > 0, "sentiment"].mean()
    bad = g.loc[g["macro_context"] < 0, "sentiment"].mean()
    rows.append({
        "source": src,
        "n_country_articles": len(g),
        "sentiment_macro_corr": corr,
        "macro_sensitivity": good - bad,
    })

sentiment_signals = pd.DataFrame(rows)
display(sentiment_signals.sort_values("macro_sensitivity", ascending=False).round(3))

In [0]:
# Composite per-source reliability: each component is scaled against FIXED, meaningful
# anchors (not min-max), so a source's score doesn't depend on the other sources and small
# gaps stay small. Anchors:
#   factual: 50% = coin-flip -> 0, 90% -> 1 (chance vs excellent on checkable claims)
#   corroboration: 0% -> 0, 20% -> 1
#   volume: articles / 750 (largest outlet ~= 1)
#   rare-event share: 0% -> 0, 25% -> 1 (a quarter of all rare events = strong)
#   sentiment sensitivity / correlation: -0.05 -> 0, +0.05 -> 1 (flat tone = 0.5)
comp = (
    structural
    .merge(acc.reset_index()[["source", "smoothed_accuracy"]], on="source", how="left")
    .merge(sentiment_signals[["source", "sentiment_macro_corr", "macro_sensitivity"]], on="source", how="left")
)


def anchored(s: pd.Series, floor: float, ceiling: float) -> pd.Series:
    return ((s.astype(float) - floor) / (ceiling - floor)).clip(0, 1)


# Sources without a stable sentiment signal get the floor (least credit), not the field minimum
comp["factual_score"] = anchored(comp["smoothed_accuracy"], 0.50, 0.90)
comp["corroboration_score"] = anchored(comp["corroboration_rate"], 0.00, 0.20)
comp["volume_score"] = anchored(comp["articles"], 0, 750)
comp["rare_score"] = anchored(comp["rare_event_share"], 0, 0.25)
comp["sensitivity_score"] = anchored(comp["macro_sensitivity"].fillna(-0.05), -0.05, 0.05)
comp["corr_score"] = anchored(comp["sentiment_macro_corr"].fillna(-0.05), -0.05, 0.05)

WEIGHTS = {
    "factual_score": 0.35,       # does the source's checkable claims match macro ground truth
    "corroboration_score": 0.10, # are its stories independently carried elsewhere
    "volume_score": 0.10,        # coverage breadth
    "rare_score": 0.15,          # does it cover disruptive events, not just filler
    "sensitivity_score": 0.15,   # does its tone move with genuinely good/bad macro weeks
    "corr_score": 0.15,          # correlation of sentiment with macro context
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9
comp["composite_reliability"] = sum(comp[k] * w for k, w in WEIGHTS.items())
comp = comp.sort_values("composite_reliability", ascending=False).reset_index(drop=True)

print(f"Weights: {WEIGHTS}")
comp = comp.merge(acc.reset_index()[["source", "n_checkable"]], on="source", how="left")
cols = ["source", "composite_reliability"] + list(WEIGHTS) + ["n_checkable"]
display(comp[cols].round(3))

In [0]:
# ML: predict whether an article's claim matches macro ground truth from sentiment, claim type, and
# TF-IDF headline features, tracked with MLflow. Features deliberately exclude macro values
# (that's the ground-truth side).
import mlflow
import mlflow.sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from mlflow.models import infer_signature

ml = news[news["checkable"]].copy()
ml["is_wire_flag"] = ml["is_wire"].map({True: "wire", False: "regional"})
ml["headline_len"] = ml["headline"].str.len()

NUMERIC = ["headline_sentiment", "body_sentiment", "headline_len"]
CATEGORICAL = ["claim", "is_wire_flag"]
TEXT = ["headline"]  # raw headline text consumed by TF-IDF inside the model pipeline
FEATURES = NUMERIC + CATEGORICAL + TEXT

# Temporal split: hold out the latest 20% of weeks (mirrors scoring future articles).
# Split on a week cutoff so rows sharing a week stay together.
ml = ml.sort_values("week").reset_index(drop=True)
cutoff_week = int(ml["week"].quantile(0.8))
train, test = ml[ml["week"] <= cutoff_week], ml[ml["week"] > cutoff_week]
X_train, y_train = train[FEATURES], train["claim_accurate"].astype(int)
X_test, y_test = test[FEATURES], test["claim_accurate"].astype(int)
print(f"train: {len(train)} rows (weeks {train['week'].min()}-{train['week'].max()}) | "
      f"test: {len(test)} rows (weeks {test['week'].min()}-{test['week'].max()})")
print(f"target balance — train: {y_train.mean():.1%} accurate | test: {y_test.mean():.1%} accurate")

class TextSelector(BaseEstimator, TransformerMixin):
    """Pass a single text column to TfidfVectorizer as a 1-D Series."""

    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.key]

    def get_feature_names_out(self, input_features=None):
        return np.array([self.key])


model = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("tfidf", Pipeline([
            ("select", TextSelector("headline")),
            ("vec", TfidfVectorizer(max_features=100, min_df=2, stop_words="english")),
        ]), TEXT),
    ])),
    ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

mlflow.set_experiment("/Users/matthew.gualino@gmail.com/news_source_reliability")
with mlflow.start_run(run_name=f"claim-accuracy-tfidf-logreg-{SOURCE_SCOPE}-v1"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "source_scope": SOURCE_SCOPE,
        "features": str(FEATURES),
        "n_train": len(train), "n_test": len(test),
        "split": "temporal last 20% of weeks",
        "tfidf_max_features": 100,
        "tfidf_min_df": 2,
        "random_state": 42,
    })
    model.fit(X_train, y_train)
    p_test = model.predict(X_test)
    metrics = {
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, p_test),
        "test_f1": f1_score(y_test, p_test),
    }
    if y_test.nunique() > 1:
        metrics["test_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
    signature = infer_signature(X_train.head(50), model.predict(X_train.head(50)))
    model_info = mlflow.sklearn.log_model(
        model, name="claim_accuracy_logreg", signature=signature, input_example=X_train.head(3)
    )
    print(f"MLflow run_id: {mlflow.active_run().info.run_id}")
    print({k: round(float(v), 3) for k, v in metrics.items()})

# Which features drive reliability: positive coefficient => feature associated with accurate claims
coef = pd.DataFrame({
    "feature": model.named_steps["preprocess"].get_feature_names_out(),
    "coefficient": model.named_steps["logreg"].coef_[0],
}).sort_values("coefficient")

non_text = coef[~coef["feature"].str.startswith("tfidf")]
print("Sentiment / claim-type features:")
display(non_text.round(3))

tfidf_coef = coef[coef["feature"].str.startswith("tfidf")].copy()
tfidf_coef["abs_coefficient"] = tfidf_coef["coefficient"].abs()
print(f"TF-IDF features ({len(tfidf_coef)} dims) — top 15 by |coefficient|:")
display(tfidf_coef.sort_values("abs_coefficient", ascending=False).head(15)[["feature", "coefficient"]].round(3))

In [0]:
# Independent audit of the World Trade Monitor factual-accuracy score.
# Self-contained: reloads the raw CSVs and re-derives every verdict via a different
# code path (dict lookup on week-1) than the pipeline's merge-based cell 3,
# so any inherited bug would show up as a mismatch.
import pandas as pd

raw_news = pd.read_csv("/Workspace/Repos/DSDG Hackathon/DSDG Hackathon/Data/news.csv")
raw_macro = pd.read_csv("/Workspace/Repos/DSDG Hackathon/DSDG Hackathon/Data/macro.csv")

NAME2CODE = {"Halvard": "HAL", "Kestria": "KES", "Valdmar": "VAL",
             "Orinoque": "ORI", "Aurelia": "AUR", "Sunda Coast": "SUN"}
WIRE_SET = {"Freight Signal", "World Trade Monitor", "Global Commodities Brief", "Merrow Maritime Daily"}

# Ground truth as a lookup table: country -> {week: (pmi, policy_rate, tension_index)}
truth = {}
for r in raw_macro.itertuples():
    truth.setdefault(r.country, {})[r.week] = (r.pmi, r.policy_rate, r.tension_index)


def audit(row):
    """Return True/False if the claim is checkable and (in)accurate, else None."""
    h, w = row["headline"], row["week"]
    code = next((c for n, c in NAME2CODE.items() if n in h), None)
    if code is None or code not in truth:
        return None
    if w == 1 or w not in truth[code] or (w - 1) not in truth[code]:
        return None  # no prior-week ground truth
    pmi, rate, tension = truth[code][w]
    pmi_p, rate_p, tension_p = truth[code][w - 1]
    if "factory activity edges up" in h:
        return pmi > pmi_p
    if "factory activity edges down" in h:
        return pmi < pmi_p
    if "holds rates steady" in h:
        return rate == rate_p
    if "Tensions rise" in h:
        return tension > tension_p
    return None


wires = raw_news[raw_news["source"].isin(WIRE_SET)].copy()
wires["audit"] = wires.apply(audit, axis=1)
aud = wires[wires["audit"].notna()].copy()
aud["audit"] = aud["audit"].astype(bool)

prior = aud["audit"].mean()
summary = aud.groupby("source")["audit"].agg(["count", "sum", "mean"]).sort_values("mean")
summary["smoothed_m10"] = (summary["count"] * summary["mean"] + 10 * prior) / (summary["count"] + 10)
print(f"INDEPENDENT RECOUNT — wire checkable claims: {len(aud)} | overall accuracy: {prior:.1%}\n")
print(summary.round(4).to_string())

wtm = summary.loc["World Trade Monitor"]
print(f"\nWorld Trade Monitor: {int(wtm['sum'])} correct of {int(wtm['count'])} = {wtm['mean']:.1%} raw | "
      f"{wtm['smoothed_m10']:.3f} smoothed (m=10 shrinkage toward the {prior:.1%} wire-group rate)")

# Compare with the pipeline's stored figures
try:
    stored = news[news["checkable"]].groupby("source")["claim_accurate"].agg(["count", "mean"])
    wtm_stored = stored.loc["World Trade Monitor"]
    match = (int(wtm_stored["count"]) == int(wtm["count"])) and abs(wtm_stored["mean"] - wtm["mean"]) < 1e-9
    print(f"Pipeline stored: {int(wtm_stored['count'])} checkable at {wtm_stored['mean']:.1%} -> "
          f"{'MATCH — score confirmed' if match else 'MISMATCH — investigate'}")
except NameError:
    print("(pipeline state unavailable; audit stands alone)")

# Where WTM loses: claim-type breakdown
wtm_rows = aud[aud["source"] == "World Trade Monitor"]
print("\nWTM accuracy by claim type:")
print(wtm_rows.groupby(
    wtm_rows["headline"].map(lambda h: "pmi_up" if "edges up" in h else
                             "pmi_down" if "edges down" in h else
                             "rate_steady" if "rates steady" in h else "tension_up")
)["audit"].agg(["count", "sum", "mean"]).round(3).to_string())

# Smoothing sensitivity: how much the m=10 choice moves the score
n, raw_rate = int(wtm["count"]), wtm["mean"]
print("\nSmoothing sensitivity for WTM (smoothed accuracy under different shrinkage strengths):")
for m_val in (0, 5, 10, 20, 40):
    print(f"  m={m_val:>2}: {(n * raw_rate + m_val * prior) / (n + m_val):.3f}")

wrong = wtm_rows[~wtm_rows["audit"]].sort_values("week")
print(f"\nSample of WTM's inaccurate claims (of {len(wrong)} total) — first 10:")
display(wrong[["week", "headline"]].head(10).reset_index(drop=True))

In [0]:
# Vocabulary reliability: which words and phrases mark reliable vs unreliable reporting?
# For every headline/body phrase appearing in >= 5 checkable articles:
#   - empirical accuracy: share of those articles whose claims matched macro.csv ground truth
#   - weight = accuracy - 71.7% baseline (positive = reliable vocabulary)
#   - model_weight: the TF-IDF logistic regression's learned coefficient, where the term is in its vocabulary
# Then profile each source by how much reliable vs unreliable vocabulary it uses.
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

chk = news[news["checkable"]].copy()
chk["text"] = (chk["headline"] + " " + chk["body"]).str.lower()
baseline = chk["claim_accurate"].mean()

cv = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
X = cv.fit_transform(chk["text"]).toarray()
phrases = np.array(cv.get_feature_names_out())
y = chk["claim_accurate"].astype(bool).values

rows = []
for i, term in enumerate(phrases):
    mask = X[:, i] > 0
    n = int(mask.sum())
    if n >= 5:
        rows.append({"phrase": term, "n_articles": n, "accuracy": float(y[mask].mean())})
vocab = pd.DataFrame(rows).sort_values("accuracy", ascending=False)

coef_w = pd.Series(model.named_steps["logreg"].coef_[0],
                   index=model.named_steps["preprocess"].get_feature_names_out())
vocab["model_weight"] = vocab["phrase"].map(lambda t: coef_w.get(f"tfidf__{t}"))
vocab["delta_vs_baseline"] = vocab["accuracy"] - baseline

# Weights for canonical terms the analysis cares about (e.g. 'central bank')
CANONICAL = ["central bank", "rates steady", "factory activity", "activity", "edges up", "edges down",
            "survey manufacturers", "inflation", "benchmark", "tensions"]
print(f"Baseline accuracy across {len(chk)} checkable claims: {baseline:.1%}\n")
print("Canonical vocabulary weights:")
display(vocab[vocab["phrase"].isin(CANONICAL)].sort_values("accuracy", ascending=False).round(3))

print("MOST RELIABLE vocabulary (accuracy >= 85%, n >= 5):")
display(vocab[vocab["accuracy"] >= 0.85].head(12).round(3))

print("MOST UNRELIABLE vocabulary (accuracy <= 60%, n >= 5):")
display(vocab[vocab["accuracy"] <= 0.60].sort_values("accuracy").head(12).round(3))

# Per-source vocabulary profile: share of each source's articles using reliable vs unreliable terms
# Generic (country-neutral) phrases only, so the per-source profile measures vocabulary
# style rather than which countries a source happens to cover
GENERIC = ~vocab["phrase"].str.contains("halvard|kestria|valdmar|orinoque|aurelia|sunda|coast|strait|sea", regex=True)
reliable_terms = vocab[(vocab["accuracy"] >= 0.85) & (vocab["n_articles"] >= 20) & GENERIC].head(12)["phrase"].tolist()
unreliable_terms = vocab[(vocab["accuracy"] <= 0.60) & (vocab["n_articles"] >= 20) & GENERIC].sort_values("accuracy").head(12)["phrase"].tolist()
rel_pat = "|".join(re.escape(t) for t in reliable_terms)
unr_pat = "|".join(re.escape(t) for t in unreliable_terms)

text_all = (news["headline"] + " " + news["body"]).str.lower()
news["has_reliable_vocab"] = text_all.str.contains(rel_pat, regex=True)
news["has_unreliable_vocab"] = text_all.str.contains(unr_pat, regex=True)

usage = news.groupby("source").agg(
    reliable_vocab_pct=("has_reliable_vocab", "mean"),
    unreliable_vocab_pct=("has_unreliable_vocab", "mean"),
).sort_values("reliable_vocab_pct", ascending=False).reset_index()
usage["vocab_bias"] = usage["reliable_vocab_pct"] - usage["unreliable_vocab_pct"]
display(usage.round(3))

print(f"\nreliable phrases used: {reliable_terms}")
print(f"unreliable phrases used: {unreliable_terms}")

## Key Findings (all 16 outlets, anchored scoring, TF-IDF model)

**Scope:** 4,317 articles from all 16 outlets, weeks 1-208. Scores use FIXED anchors, not min-max, so they don't depend on the other sources and small gaps stay small (WTM's factual gap is now 0.34, not 0.0).

**Factual accuracy (516 checkable claims, macro.csv ground truth; overall 71.7%):**
- Central-bank "holds rates steady" claims: **93.1% accurate** (261 checked) — the one reliably reported fact
- "Factory activity edges up/down" claims: **48% combined** (250 checked) — a coin flip everywhere; includes the week-6 contradiction where Freight Signal printed both directions for Sunda Coast
- "Tensions rise" claims: 4 of 5 accurate
- **World Trade Monitor audit (cell 8, independent recount): 52 of 83 = 62.7% confirmed by a separate code path.** Its losses concentrate in "edges down" claims: only 3 of 18 accurate (16.7%) — it reports downturns when PMI actually rose. Smoothing choice is immaterial (0.627-0.649 across m=0-40).
- Regional papers score higher on accuracy (raw 64-92%) but on only 10-18 checkable claims each — treat those scores as indicative (see `n_checkable` column in cell 6).

**Composite reliability ranking (fixed anchors; weights: factual 35%, rare-event 15%, sentiment sensitivity 15%, sentiment-macro correlation 15%, corroboration 10%, volume 10%):**
- Wires lead on breadth + rare events: **Global Commodities Brief 0.68**, **Freight Signal 0.67** (all 5 of its rare events), **Merrow Maritime Daily 0.60**
- **World Trade Monitor 0.50** — a mid-table score now: strong on rare events (0.8) and volume, weak on factual (0.34) and sentiment alignment (0.23), consistent with the audited false-pessimism
- Best regional papers: Kestria Daily 0.58, Sunda Business Times 0.57, Aurelian Ledger 0.54, Northern Capital Times 0.53 — mostly rate-steady claims, good corroboration, but little rare-event coverage and small samples
- Bottom tier: Halvard Economic Review 0.38, Coastal Observer 0.34, Halvard Courier 0.33, Valdmar Herald 0.31, Orinoque Mining Journal 0.29 — no rare-event coverage, flat or wrong-directed tone (Valdmar Herald and Orinoque Mining Journal have negative sentiment-macro correlations), and thin samples

**Vocabulary reliability weights (cell 9):**
- Most reliable vocabulary: `central bank`, `benchmark`, `rates steady`, `inflation easing`, `kept`/`holds`/`unchanged` — **93.1%** of the 261 articles carrying these terms matched macro ground truth (+0.21 above the 71.7% baseline)
- Most unreliable vocabulary: `factory activity`, `survey of manufacturers`, `edges`, `moving slightly`, `showed` — **49.2%** of 250 articles (−0.23 below baseline; the TF-IDF model's learned weight for `activity` is −0.40)
- Country-specific: Kestria factory-activity reporting is the worst phrase cluster (32.4% accurate); Halvard/Aurelia/Kestria rate-decision coverage is near-perfect (95-100%)
- Per-source vocabulary profile (generic terms only): every source leans toward unreliable vocabulary (survey/activity stories outnumber rate stories); Halvard Economic Review is the most skewed (38.8% of its articles use unreliable vocabulary vs 14.9% reliable, bias −0.24); Aurelia Today (−0.02) and Orinoque Mining Journal (−0.04) are most balanced; the four wires cluster at −0.06 to −0.11

**ML claim-accuracy model with TF-IDF headline features (MLflow run `a652885416fd48abb229dbee8898c58e`, all-scope):**
- Test accuracy 71.8%, AUC 0.744, F1 0.791 (temporal holdout: 413 train rows, weeks 2-161; 103 test rows, weeks 162-208) — best AUC of the three pipeline versions
- Set `SOURCE_SCOPE = "wires"` and re-run to reproduce the audited four-wire deep-dive (prior run `ee9c32b433bd4dcc8d74701408e1861c`)

**Caveats:** sentiment uses the custom finance/news lexicon (VADER unavailable on this compute); regional papers' component scores rest on 10-18 checkable claims (shrinkage-smoothed toward the 71.7% global rate — their true accuracy is less certain than the wires'); the TF-IDF vocabulary is small because headlines are templated; sentiment-macro correlation/sensitivity anchors (±0.05) mean even the best tone-tracking sources score only ~0.6-0.9.